In [3]:
%pip install datasets

  Using cached pyarrow-26.0.1-cp314-cp314-macosx_12_0_arm64.whl.metadata (3.0 kB)
  Using cached huggingface_hub-2.4.0-py3-none-any.whl.metadata (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.3/608.3 kB 1.4 MB/s  0:00:002.1 MB/s eta 0:00:01
Using cached huggingface_hub-2.4.0-py3-none-any.whl (857 kB)
Using cached pyarrow-26.0.1-cp314-cp314-macosx_12_0_arm64.whl (37.3 MB)
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: huggingface-hub━━━━╸━━━━━━━━━━━━━━━━━ 4/7 [httpx2]]
    Found existing installation: huggingface_hub 1.29.08;114m╸━━━━━━━━━━━━━━━━━ 4/7 [httpx2]
    Uninstalling huggingface_hub-1.29.0:╸━━━━━━━━━━━━━━━━━ 4/7 [httpx2]
      Successfully uninstalled huggingface_hub-1.29.0;38;114m╸━━━━━━━━━━━━━━━━━ 4/7 [httpx2]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [datasets]━━ 6/7 [datasets]ggingface-hub]
ERROR: pip's dependency resolver d

In [11]:
from datasets import load_dataset
ds = load_dataset("xmarva/tfs-tatoeba-en-ru")

In [43]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [12]:
print(ds)
print(ds['train'][0])

DatasetDict({
    train: Dataset({
        features: ['en', 'ru'],
        num_rows: 508093
    })
    validation: Dataset({
        features: ['en', 'ru'],
        num_rows: 10370
    })
    test: Dataset({
        features: ['en', 'ru'],
        num_rows: 36385
    })
})
{'en': 'You should have accepted his advice.', 'ru': 'Тебе стоило последовать его совету.'}


In [13]:
import unicodedata
def clean(text):
    return " ".join(unicodedata.normalize("NFKC", text).split())
def normalize(ex):
    return {'en': clean(ex['en']), 'ru': clean(ex['ru'])}
ds = ds.map(normalize)
print(ds)

DatasetDict({
    train: Dataset({
        features: ['en', 'ru'],
        num_rows: 508093
    })
    validation: Dataset({
        features: ['en', 'ru'],
        num_rows: 10370
    })
    test: Dataset({
        features: ['en', 'ru'],
        num_rows: 36385
    })
})


In [14]:
def non_empty(ex):
    return len(ex['en'].strip()) > 0 and len(ex['ru'].strip()) > 0

ds = ds.filter(non_empty)
print(ds)

DatasetDict({
    train: Dataset({
        features: ['en', 'ru'],
        num_rows: 508093
    })
    validation: Dataset({
        features: ['en', 'ru'],
        num_rows: 10370
    })
    test: Dataset({
        features: ['en', 'ru'],
        num_rows: 36385
    })
})


In [15]:
def is_same(ex):
    return False if ex['en'] == ex['ru'] else True

ds = ds.filter(is_same)
print(ds)

DatasetDict({
    train: Dataset({
        features: ['en', 'ru'],
        num_rows: 508093
    })
    validation: Dataset({
        features: ['en', 'ru'],
        num_rows: 10370
    })
    test: Dataset({
        features: ['en', 'ru'],
        num_rows: 36384
    })
})


In [16]:
def cyr_share(text):
    letters = [ch for ch in text if ch.isalpha()]
    if not letters:
        return 0.0
    cyr = [ch for ch in letters if 'а' <= ch.lower() <= 'я' or ch.lower() == 'ё']
    return len(cyr) / len(letters)

def alphabet(ex):
    return cyr_share(ex["ru"]) > 0.5 and cyr_share(ex["en"]) < 0.1

ds = ds.filter(alphabet)
print(ds)

DatasetDict({
    train: Dataset({
        features: ['en', 'ru'],
        num_rows: 508075
    })
    validation: Dataset({
        features: ['en', 'ru'],
        num_rows: 10370
    })
    test: Dataset({
        features: ['en', 'ru'],
        num_rows: 36380
    })
})


In [17]:
def length(ex):
    n_en = len(ex['en'].split())
    n_ru = len(ex['ru'].split())
    return 1 <= n_en <= 40 and 1 <= n_ru <= 40

ds['train'] = ds['train'].filter(length)
print(ds)

DatasetDict({
    train: Dataset({
        features: ['en', 'ru'],
        num_rows: 508075
    })
    validation: Dataset({
        features: ['en', 'ru'],
        num_rows: 10370
    })
    test: Dataset({
        features: ['en', 'ru'],
        num_rows: 36380
    })
})


In [18]:
def key(s):
    return s.lower().strip(".?!")

held_out = {key(s) for s in ds["validation"]["en"]} | {key(s) for s in ds["test"]["en"]}
train = ds["train"].filter(lambda ex: key(ex["en"]) not in held_out)
print(train.shape)
ds["train"] = train

(508012, 2)


In [19]:
seen = set()

def first_seen(ex):
    pair = (ex["en"], ex["ru"])
    if pair in seen:
        return False
    seen.add(pair)
    return True

train = ds["train"].filter(first_seen)
print(len(ds["train"]), len(train))

508012 508012


In [20]:
ds["train"] = train
ds.save_to_disk("data/clean")

Saving the dataset (0/1 shards):   0%|          | 0/508012 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10370 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/36380 [00:00<?, ? examples/s]